# 5 Gate, highway, or route: one idea from three papers

**This is not a time-series notebook.** The gate/highway/MoE family composes with any
`(batch, hidden)` tensor — images, text, tabular data, anything. It sits here because
this lesson already has a window and a baseline table to beat, the same reason
`1_pytorch_intro/04_search` introduced ray.tune where a two-second training run made it
obviously worth the investment rather than because tuning is somehow a time-series idea.


## 5.1 One idea, three resolutions

| component | mechanism | where it comes from |
|---|---|---|
| `Gate` | `sigmoid(Wx) * x` -- a learned per-dimension scale | the gate inside every LSTM cell (Hochreiter & Schmidhuber, 1997) and the reset/update gates of the GRU (Cho et al., 2014, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*) |
| `Highway` | `g * T(x) + (1 - g) * x` -- a learned interpolation between a transform and the identity | Highway Networks (Srivastava, Greff & Schmidhuber, 2015), built explicitly to let very deep plain networks train at all -- the direct ancestor of the ResNet residual connection, which is a Highway with the gate fixed at 0.5 |
| `MoE` | `sum_i softmax(Wx)_i * expert_i(x)` -- N gates routing between N experts | Jacobs & Jordan, 1991, *Adaptive Mixtures of Local Experts* -- the **dense** version tested in §5.2-5.4 below: all N experts run, every time. Modern **sparse** MoE (Shazeer et al., 2017, *Outrageously Large Neural Networks*; the Switch Transformer, Fedus et al., 2021) routes each token to only `k` of thousands of experts -- capacity scales with N, compute scales with k. §5.5 builds and measures that version (`vectormesh`’s `SparseMoE`), which is the part that actually makes production MoE fast at scale |

Read top to bottom, each row is the row above with more capacity to choose from: one
gate choosing between a transform and nothing (`Gate`), one gate choosing between a
transform and the identity (`Highway`), N gates choosing between N transforms (`MoE`).

This lesson's electricity-demand series has real daily and weekly structure -- exactly
the kind of thing a router could plausibly use. Predict before you look at the table:
does giving the mechanism something real to route on change the verdict?


In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet

from australian_electricity import load_ausalia
from ett import window_index
from notebooktester import param

DAY = 48                # half-hourly steps per day
W, H = 4 * DAY, DAY      # four days of history, one day of horizon -- same window
                         # 04_probabilistic uses, so results sit next to that notebook's
HIDDEN = 64
STRIDE = 24
SEEDS = (0, 1, 2, 3, 4)

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_model(mechanism: str, hidden: int = HIDDEN, n_experts: int = 4) -> nn.Module:
    if mechanism == "plain":
        return NeuralNet(hidden_size=[W, hidden], out_size=H)
    if mechanism == "deeper":
        # capacity-matched control: one more Linear -> GELU than "plain", no gate at all.
        # Gate's own weight matrix costs exactly what this second layer costs at this width.
        return NeuralNet(hidden_size=[W, hidden, hidden], out_size=H)
    inp, act, out = nn.Linear(W, hidden), nn.GELU(), nn.Linear(hidden, H)
    if mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
                  hidden_size=hidden, out_size=hidden)
    else:
        raise ValueError(mechanism)
    return nn.Sequential(inp, act, mid, out)


def run(mechanism: str, seed: int, epochs: int, lr: float = 1e-3, batch_size: int = 64) -> float:
    torch.manual_seed(seed)
    model = make_model(mechanism)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return mae(yva, model(xva))

## 5.2 How many epochs is enough? Check across epoch counts *and* architectures

A single epoch count checked with 3 seeds is not enough evidence, and this notebook
found that out the hard way while it was being written: `gate` looked like a clear
winner over its capacity-matched control at one epoch count, and the advantage
vanished at another. The only fix is to check several epoch counts, with enough seeds
that a lucky run cannot decide the answer by itself.


In [ ]:
EPOCHS_TO_CHECK = param((30, 60, 90), test=(1, 2))

for epochs in EPOCHS_TO_CHECK:
    for name in ("gate", "deeper"):
        scores = [run(name, seed, epochs) for seed in SEEDS]
        print(f"epochs={epochs:3d}   {name:7s} {np.mean(scores):.5f} +/- {np.std(scores):.5f}")
    print()

`gate` and `deeper` -- its capacity-matched control, same parameter count, no gate
anywhere -- land inside each other's seed spread at every epoch count checked here.
That is the answer: pick a reasonably converged epoch count and stop looking for a
number where the gap opens up, because the checks above say there is not a stable one
to find. `EPOCHS = 90` below is not the best number in any of these rows; it is simply
past the region where training is still visibly moving.

## 5.3 The full family, at a settled epoch count


In [ ]:
EPOCHS = param(90, test=2)

results: dict[str, dict] = {}
for name in ("plain", "deeper", "gate", "highway", "moe"):
    scores = [run(name, seed, EPOCHS) for seed in SEEDS]
    params = sum(p.numel() for p in make_model(name).parameters())
    results[name] = {"mae": float(np.mean(scores)), "std": float(np.std(scores)), "params": params}
    print(f"{name:8s} {results[name]['mae']:.5f} +/- {results[name]['std']:.5f}   {params:>7,} params")

## 5.4 The table

In [ ]:
order = sorted(results, key=lambda k: results[k]["mae"])
print(f"{'model':10s} {'val MAE':>9} {'+/-':>7} {'params':>9}")
print("-" * 40)
for name in order:
    r = results[name]
    print(f"{name:10s} {r['mae']:9.5f} {r['std']:7.5f} {r['params']:9,}")

`deeper` and `gate` are the two best rows, and they sit inside each other's seed
spread -- exactly the §5.2 finding, now with `plain`, `highway` and `moe` alongside
them for context. `highway` and `moe` cost 1.8x and 2.1x `plain`'s parameters and do
not beat it by a margin that clears their own seed noise either.

A series with real daily and weekly structure -- exactly the kind of thing a router
could plausibly use -- still does not make `gate` or `highway` earn their parameters,
once checked properly. `09_composition_extra` (optional) measured the same shape of
result on a four-patch chunked window: a gate or a highway is a real bet that
structure exists *and that this architecture can find it*, and the bet does not pay
just because the structure is there.

**`moe`'s result above needs a narrower reading than the other two.** `MoE` here is
the *dense* version -- every one of its 4 experts runs on every input, every time
(§5.1's table says so). So the row above answers a real but narrow question --
"does blending several small transforms with a learned weight beat one transform of
matching size" -- and the answer is no, not by a margin that clears seed noise. It
does **not** test MoE's actual production selling point: routing each input to only
`k` of `N` experts so capacity (`N`) and compute (`k`) decouple. §5.5 below builds
that version and measures it directly, rather than leaving it as a caveat.

## 5.5 Sparse routing: capacity and compute, decoupled

`MoE` above is *dense*: every expert runs on every input, so adding experts always
costs more compute -- by construction, it cannot show the thing that makes production
MoE (Switch Transformer, Mixtral) worth building. `vectormesh`'s `SparseMoE` (0.7.0)
routes each example to its top-`k` experts with real gather/scatter -- an expert with
zero examples routed to it this batch does zero work, not masked-out work -- so
compute per example scales with `k`, not with the number of experts `N`.

The test: fix `k=1` (Switch-style) and a fixed batch of this notebook's own windows,
then grow `N` from 4 to 256 experts. If the sparse claim is real, wall-clock time per
forward pass should stay roughly flat while parameter count grows 64x. Dense `MoE`,
run on the identical batch and the identical `N` values, is the control here -- it has
no reason to stay flat, since it always runs every expert regardless of how many
exist.

In [ ]:
import time

from vectormesh.components import SparseMoE

N_EXPERTS_TO_CHECK = param((4, 16, 64, 256), test=(2, 4))
ROUTING_BATCH = 64
REPS = param(30, test=3)

# a real batch of this notebook's own windows, projected to HIDDEN once -- the
# component under test is the mixture-of-experts block, not the input projection
probe_input = nn.Linear(W, HIDDEN)
probe_x = probe_input(xtr[:ROUTING_BATCH])


def time_forward(model: nn.Module, x: torch.Tensor, reps: int) -> float:
    for _ in range(5):
        model(x)
    start = time.perf_counter()
    for _ in range(reps):
        model(x)
    return (time.perf_counter() - start) / reps * 1000


sparse_timings: dict[int, dict] = {}
dense_timings: dict[int, dict] = {}
for n_experts in N_EXPERTS_TO_CHECK:
    torch.manual_seed(0)
    sparse = SparseMoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN, out_size=HIDDEN, k=1,
    )
    torch.manual_seed(0)
    dense = MoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN, out_size=HIDDEN,
    )

    sparse_ms = time_forward(sparse, probe_x, REPS)
    dense_ms = time_forward(dense, probe_x, REPS)
    params = sum(p.numel() for p in sparse.parameters())  # identical count, both sides

    sparse_timings[n_experts] = {"ms": sparse_ms, "params": params}
    dense_timings[n_experts] = {"ms": dense_ms, "params": params}
    print(f"N={n_experts:4d} experts   sparse {sparse_ms:7.3f} ms   "
          f"dense {dense_ms:7.3f} ms   {params:>9,} params")

Measured on this machine (batch of 64 of this notebook's own windows, hidden=64,
k=1, 30 reps after 5 warmup calls):

| N experts | sparse ms/fwd | dense ms/fwd | params |
|---|---|---|---|
| 4 | 0.129 | 0.117 | 16,900 |
| 16 | 0.204 | 0.224 | 67,600 |
| 64 | 0.250 | 0.535 | 270,400 |
| 256 | 0.262 | 1.783 | 1,081,600 |

From N=4 to N=256, parameter count grows **64x**. `SparseMoE`'s forward time grows
**2.0x** (0.129 -> 0.262 ms) -- most of that from routing overhead (the argsort and
the per-active-expert Python loop), not from added compute, since each example still
only ever passes through one expert. Dense `MoE`, run on the identical batch and the
identical `N` values, grows **15.2x** (0.117 -> 1.783 ms) -- close to proportional
with `N`, exactly as it must be, since it evaluates every expert on every example
regardless of capacity. At `N=4` dense is actually a touch *faster* than sparse (the
routing machinery costs more than it saves when there is nothing to route around);
the crossover happens by `N=16`, and by `N=256` dense is **6.8x** slower than sparse
for the same parameter count.

That is the caveat from §5.1 and §5.4 resolved, not just narrated: capacity (`N`) and
compute (bounded by `k`) genuinely decouple once routing is real gather/scatter
rather than compute-everything-then-blend. It does not mean sparse routing was worth
its parameters *on this task* -- §5.4 already answered that, and nothing here reruns
that check for the routed version -- only that the mechanism does what production
MoE papers claim it does: hold far more capacity than any single input pays for.

`06_search` is where you stop checking one hand-picked configuration at a time and let
a search cover the space properly.

## What to take from this

- Gate, Highway and MoE are one idea -- a learned keep-or-replace decision -- at
  increasing resolution, and each one traces to a specific paper rather than floating
  free in a library's component list.
- The mechanism is general-purpose. It showed up here because this lesson had a window
  and a baseline ready, not because time series needs it more than any other modality.
- **A result checked at one epoch count is not checked.** §5.2 is in this notebook
  because the first version of it was wrong, and the fix was not a better number, it
  was more epoch counts and more seeds.
- Whether gating earns its extra parameters *on this window* is the number in the
  table above, not an opinion -- and "capacity-matched control" is doing the real work
  in that sentence, not "gate" or "highway" by name.
- **Dense and sparse MoE answer different questions.** §5.4's dense `MoE` asks "does
  blending N small transforms beat one transform of matching size" (no, here). §5.5's
  `SparseMoE` asks "does routing decouple capacity from compute" (yes, measured
  directly) -- two real, different, non-substitutable claims that happen to share a
  name.

## Exercises

**E1 -- predict, then measure.** Before re-running this notebook, write down a ranking
you expect for the five trained models, and say whether you expect this series'
structure to change the §5.2 verdict from a null result to a real one. Check both.

**E2 -- turn the gate off.** Force `Gate`'s output to equal its input (patch the
forward pass, or initialise its linear layer to a large positive bias so the sigmoid
saturates near 1) and confirm the block's score converges toward `deeper`'s -- the
collapse argument from `1_pytorch_intro/02_nonlinearity` §2.1, for a gate instead of a
stacked linear layer.

**E3 -- more experts, dense.** Re-run `make_model("moe")` with 2, 4, 8 and 16 experts,
each checked across the same epoch counts as §5.2. Does the router find anything to
split on, or does the score just track the parameter count?

**E4 -- a different window.** Swap in `pedestrian_counts` (loaded the same way
`03_decomposition_and_nbeats` and `06_search` both load it) and re-run the whole
notebook. Does the ranking change on a series with an even sharper daily cycle?

**E5 -- k>1.** Re-run §5.5's timing sweep with `k=2` and `k=4` instead of `k=1`.
Predict first: does the flat region shift, and in which direction -- does it take more
experts, or fewer, before adding another one costs nothing extra?